# HealSense: LSTM Health Risk Prediction

**Production-ready model for real-time health monitoring**

This notebook trains an LSTM model to classify health status (Normal/Warning/Critical) from time-series vital signs data.

## What we're building:
- **Input**: 60 minutes of vital signs (HR, SpO₂, Temp, BP)
- **Output**: Health risk level + confidence score
- **Deployment**: Mobile-ready TFLite model (<5MB)

---

## 1. Setup & Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

# Preprocessing
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Evaluation
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support, roc_auc_score,
    roc_curve, auc
)

# Utils
import joblib

print(f"TensorFlow: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")
print(f"\n✓ All imports successful")

## 2. Configuration

In [ ]:
# Model hyperparameters
CONFIG = {
    'sequence_length': 60,  # 60 minutes of history
    'features': ['heart_rate', 'spo2', 'temperature', 'systolic_bp', 'diastolic_bp'],
    'num_features': 5,
    'num_classes': 3,  # normal, warning, critical
    'batch_size': 32,
    'epochs': 100,
    'learning_rate': 0.001,
    'lstm_units': [128, 64],
    'dropout_rate': [0.3, 0.2, 0.1],
    'dense_units': 32,
    'random_seed': 42
}

# Paths
DATA_PATH = '../data/raw/synthetic/synthetic_vital_signs.csv'
MODEL_DIR = '../data/models'
CHECKPOINT_PATH = f"{MODEL_DIR}/checkpoints/lstm_best.h5"
TFLITE_PATH = f"{MODEL_DIR}/tflite/health_model.tflite"

# Set seeds for reproducibility
np.random.seed(CONFIG['random_seed'])
tf.random.set_seed(CONFIG['random_seed'])

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 3. Load & Explore Data

In [ ]:
# Load dataset
df = pd.read_csv(DATA_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
display(df.head())

print(f"\n📊 Class Distribution:")
status_dist = df['status'].value_counts()
for status, count in status_dist.items():
    pct = (count / len(df)) * 100
    print(f"  {status.capitalize():10s}: {count:5d} ({pct:5.2f}%)")

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
status_dist.plot(kind='bar', ax=axes[0], color=['#4CAF50', '#FFC107', '#F44336'])
axes[0].set_title('Health Status Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
colors = ['#4CAF50', '#FFC107', '#F44336']
axes[1].pie(status_dist, labels=status_dist.index, autopct='%1.1f%%', 
            colors=colors, startangle=90)
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("⚠️ Notice: Class imbalance detected (Critical is underrepresented)")
print("   → We'll use weighted loss function to handle this")

## 4. Data Preprocessing

In [ ]:
def create_sequences(data, sequence_length=60):
    """
    Convert time-series data into sequences for LSTM training
    
    Args:
        data: DataFrame grouped by patient_id
        sequence_length: Number of timesteps per sequence
    
    Returns:
        X: Array of shape (samples, sequence_length, num_features)
        y: Array of labels
    """
    X, y = [], []
    
    for patient_id in data['patient_id'].unique():
        patient_data = data[data['patient_id'] == patient_id]
        
        # Extract features and labels
        features = patient_data[CONFIG['features']].values
        labels = patient_data['status'].values
        
        # Create sliding windows
        for i in range(len(features) - sequence_length + 1):
            X.append(features[i:i + sequence_length])
            # Label is the status at the END of the sequence (most recent)
            y.append(labels[i + sequence_length - 1])
    
    return np.array(X), np.array(y)

print("Creating sequences...")
X, y = create_sequences(df, CONFIG['sequence_length'])

print(f"\nSequence shape: {X.shape}")
print(f"  - Samples: {X.shape[0]}")
print(f"  - Timesteps: {X.shape[1]}")
print(f"  - Features: {X.shape[2]}")
print(f"\nLabels shape: {y.shape}")

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded, num_classes=CONFIG['num_classes'])

print("Label mapping:")
for idx, label in enumerate(label_encoder.classes_):
    print(f"  {idx}: {label}")

# Train-test split (stratified to preserve class distribution)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_categorical, 
    test_size=0.2, 
    random_state=CONFIG['random_seed'],
    stratify=y_encoded
)

print(f"\nTrain set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# CRITICAL: Normalize features (LSTM needs scaled inputs)
print("Normalizing features...")
print("\nBefore normalization:")
print(pd.DataFrame(X_train[0], columns=CONFIG['features']).describe())

# Fit scaler on training data only
scaler = StandardScaler()

# Reshape for scaling: (samples * timesteps, features)
n_samples_train, n_timesteps, n_features = X_train.shape
X_train_reshaped = X_train.reshape(-1, n_features)
X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_train_scaled = X_train_scaled.reshape(n_samples_train, n_timesteps, n_features)

# Transform test data
n_samples_test = X_test.shape[0]
X_test_reshaped = X_test.reshape(-1, n_features)
X_test_scaled = scaler.transform(X_test_reshaped)
X_test_scaled = X_test_scaled.reshape(n_samples_test, n_timesteps, n_features)

print("\nAfter normalization (scaled to mean=0, std=1):")
print(pd.DataFrame(X_train_scaled[0], columns=CONFIG['features']).describe())

print("\n✓ Feature scaling complete")
print("  (This prevents large-scale features from dominating gradients)")

In [ ]:
# Compute class weights (handle imbalance)
y_train_labels = np.argmax(y_train, axis=1)
class_weights_array = compute_class_weight(
    'balanced', 
    classes=np.unique(y_train_labels), 
    y=y_train_labels
)
class_weights = dict(enumerate(class_weights_array))

print("Class weights (to balance training):")
for idx, weight in class_weights.items():
    label = label_encoder.classes_[idx]
    print(f"  {label.capitalize():10s}: {weight:.3f}")

print("\n⚠️ Critical class has highest weight → model will focus on not missing emergencies")

## 5. Build LSTM Model

In [ ]:
def build_lstm_model(config):
    """
    Build LSTM model with dropout for regularization
    """
    model = Sequential([
        # Input layer
        layers.Input(shape=(config['sequence_length'], config['num_features'])),
        
        # LSTM Layer 1 with dropout
        layers.LSTM(config['lstm_units'][0], return_sequences=True),
        layers.Dropout(config['dropout_rate'][0]),
        
        # LSTM Layer 2 with dropout
        layers.LSTM(config['lstm_units'][1]),
        layers.Dropout(config['dropout_rate'][1]),
        
        # Dense layer
        layers.Dense(config['dense_units'], activation='relu'),
        layers.Dropout(config['dropout_rate'][2]),
        
        # Output layer (3 classes: normal, warning, critical)
        layers.Dense(config['num_classes'], activation='softmax')
    ])
    
    return model

# Build model
model = build_lstm_model(CONFIG)

# Compile
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG['learning_rate']),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
)

print("Model Architecture:")
model.summary()

# Calculate model size
trainable_params = np.sum([np.prod(v.get_shape()) for v in model.trainable_weights])
print(f"\nTrainable parameters: {trainable_params:,}")
print(f"Estimated model size: ~{(trainable_params * 4) / (1024**2):.2f} MB (float32)")

## 6. Training with Callbacks

In [ ]:
# Setup callbacks
callbacks = [
    # Save best model based on validation loss
    ModelCheckpoint(
        CHECKPOINT_PATH,
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    ),
    
    # Stop training if no improvement for 15 epochs
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Reduce learning rate when plateau
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

print("Callbacks configured:")
print("  ✓ ModelCheckpoint: Save best model")
print("  ✓ EarlyStopping: Prevent overfitting")
print("  ✓ ReduceLROnPlateau: Adaptive learning rate")

In [ ]:
# Train model
print("\n🚀 Starting training...\n")
print("="*60)

history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=CONFIG['epochs'],
    batch_size=CONFIG['batch_size'],
    class_weight=class_weights,  # Handle class imbalance
    callbacks=callbacks,
    verbose=1
)

print("="*60)
print("\n✓ Training complete!")

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss
axes[0, 0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0, 0].set_title('Model Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0, 1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0, 1].set_title('Model Accuracy', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Precision
axes[1, 0].plot(history.history['precision'], label='Train Precision', linewidth=2)
axes[1, 0].plot(history.history['val_precision'], label='Val Precision', linewidth=2)
axes[1, 0].set_title('Model Precision', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Recall
axes[1, 1].plot(history.history['recall'], label='Train Recall', linewidth=2)
axes[1, 1].plot(history.history['val_recall'], label='Val Recall', linewidth=2)
axes[1, 1].set_title('Model Recall', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Model Evaluation

In [ ]:
# Load best model
model = keras.models.load_model(CHECKPOINT_PATH)
print("✓ Loaded best model from checkpoint\n")

# Predictions
y_pred_probs = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

# Overall metrics
test_loss, test_acc, test_prec, test_rec = model.evaluate(X_test_scaled, y_test, verbose=0)

print("📊 Test Set Performance:")
print("="*50)
print(f"  Accuracy:  {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"  Precision: {test_prec:.4f}")
print(f"  Recall:    {test_rec:.4f}")
print(f"  Loss:      {test_loss:.4f}")

In [ ]:
# Detailed classification report
print("\n📋 Classification Report:")
print("="*70)
print(classification_report(
    y_true, y_pred, 
    target_names=label_encoder.classes_,
    digits=4
))

# Per-class metrics
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None
)

print("\n🎯 Per-Class Performance:")
print("="*70)
for idx, label in enumerate(label_encoder.classes_):
    print(f"\n{label.upper()}:")
    print(f"  Precision: {precision[idx]:.4f}")
    print(f"  Recall:    {recall[idx]:.4f}")
    print(f"  F1-Score:  {f1[idx]:.4f}")
    print(f"  Support:   {support[idx]}")

# Critical class check
critical_idx = list(label_encoder.classes_).index('critical')
if recall[critical_idx] < 0.90:
    print("\n⚠️ WARNING: Critical recall below 90%! Consider adjusting thresholds.")
else:
    print(f"\n✓ Critical recall: {recall[critical_idx]:.2%} (Good - catching most emergencies)")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

# Normalized confusion matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            cbar_kws={'label': 'Percentage'})
plt.title('Normalized Confusion Matrix', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves (one-vs-rest)
from sklearn.preprocessing import label_binarize

y_test_bin = label_binarize(y_true, classes=[0, 1, 2])
n_classes = y_test_bin.shape[1]

# Compute ROC curve for each class
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot
plt.figure(figsize=(10, 8))
colors = ['#4CAF50', '#FFC107', '#F44336']
for i, color, label in zip(range(n_classes), colors, label_encoder.classes_):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f'{label.capitalize()} (AUC = {roc_auc[i]:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves (One-vs-Rest)', fontsize=16, fontweight='bold', pad=20)
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nROC-AUC Scores:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {label.capitalize():10s}: {roc_auc[i]:.4f}")

## 8. Save Model Artifacts

In [ ]:
# Save preprocessing artifacts
print("Saving model artifacts...\n")

# Save scaler
scaler_path = f"{MODEL_DIR}/scaler.pkl"
joblib.dump(scaler, scaler_path)
print(f"✓ Scaler saved: {scaler_path}")

# Save label encoder
encoder_path = f"{MODEL_DIR}/label_encoder.pkl"
joblib.dump(label_encoder, encoder_path)
print(f"✓ Label encoder saved: {encoder_path}")

# Save model metadata
metadata = {
    'timestamp': datetime.now().isoformat(),
    'model_type': 'LSTM',
    'version': '1.0.0',
    'config': CONFIG,
    'performance': {
        'test_accuracy': float(test_acc),
        'test_precision': float(test_prec),
        'test_recall': float(test_rec),
        'test_loss': float(test_loss),
        'per_class': {
            label_encoder.classes_[i]: {
                'precision': float(precision[i]),
                'recall': float(recall[i]),
                'f1_score': float(f1[i]),
                'roc_auc': float(roc_auc[i])
            }
            for i in range(len(label_encoder.classes_))
        }
    },
    'classes': label_encoder.classes_.tolist(),
    'features': CONFIG['features'],
    'training_samples': int(X_train.shape[0]),
    'test_samples': int(X_test.shape[0])
}

metadata_path = f"{MODEL_DIR}/metadata/model_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Metadata saved: {metadata_path}")

print("\n✓ All artifacts saved successfully!")

## 9. TensorFlow Lite Conversion

In [ ]:
# Convert to TFLite with quantization
print("Converting model to TensorFlow Lite...\n")

converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Optimization: Quantize to float16 (smaller size, minimal accuracy loss)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

# Save TFLite model
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

# Get file sizes
import os
h5_size = os.path.getsize(CHECKPOINT_PATH) / (1024**2)
tflite_size = os.path.getsize(TFLITE_PATH) / (1024**2)
compression_ratio = (1 - tflite_size/h5_size) * 100

print("✓ TFLite conversion complete!\n")
print(f"Model sizes:")
print(f"  Original (.h5):  {h5_size:.2f} MB")
print(f"  TFLite (float16): {tflite_size:.2f} MB")
print(f"  Compression:      {compression_ratio:.1f}% smaller")
print(f"\nMobile deployment target: < 5 MB → {'✓ PASS' if tflite_size < 5 else '✗ FAIL'}")

In [ ]:
# Test TFLite model inference
print("Testing TFLite model...\n")

interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

# Get input/output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input shape:", input_details[0]['shape'])
print("Output shape:", output_details[0]['shape'])

# Test inference on first test sample
test_sample = X_test_scaled[0:1].astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_sample)

import time
start = time.time()
interpreter.invoke()
latency = (time.time() - start) * 1000

tflite_output = interpreter.get_tensor(output_details[0]['index'])
tflite_pred = np.argmax(tflite_output[0])

print(f"\nInference latency: {latency:.2f} ms")
print(f"Target: < 100 ms → {'✓ PASS' if latency < 100 else '✗ FAIL'}")

# Compare with original model
original_pred = np.argmax(y_pred_probs[0])
print(f"\nPrediction comparison:")
print(f"  Original model: {label_encoder.classes_[original_pred]}")
print(f"  TFLite model:   {label_encoder.classes_[tflite_pred]}")
print(f"  Match: {'✓' if original_pred == tflite_pred else '✗'}")

In [ ]:
# Validate TFLite accuracy on full test set
print("Validating TFLite model on full test set...\n")

tflite_predictions = []

for i in range(len(X_test_scaled)):
    test_sample = X_test_scaled[i:i+1].astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], test_sample)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    tflite_predictions.append(np.argmax(output[0]))

tflite_predictions = np.array(tflite_predictions)

# Calculate accuracy
tflite_accuracy = np.mean(tflite_predictions == y_true)

print(f"TFLite Model Test Accuracy: {tflite_accuracy:.4f} ({tflite_accuracy*100:.2f}%)")
print(f"Original Model Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Accuracy difference: {abs(tflite_accuracy - test_acc):.4f}")

if abs(tflite_accuracy - test_acc) < 0.02:
    print("\n✓ TFLite model maintains accuracy (< 2% difference)")
else:
    print("\n⚠️ Warning: TFLite accuracy differs significantly from original")

## 10. Deployment Summary

In [ ]:
print("="*70)
print("🎉 HEALSENSE MODEL TRAINING COMPLETE")
print("="*70)

print("\n📦 Model Artifacts:")
print(f"  1. Keras Model:     {CHECKPOINT_PATH}")
print(f"  2. TFLite Model:    {TFLITE_PATH}")
print(f"  3. Scaler:          {scaler_path}")
print(f"  4. Label Encoder:   {encoder_path}")
print(f"  5. Metadata:        {metadata_path}")

print("\n📊 Final Performance:")
print(f"  Test Accuracy:   {test_acc*100:.2f}%")
print(f"  Test Precision:  {test_prec:.3f}")
print(f"  Test Recall:     {test_rec:.3f}")
print(f"  Critical Recall: {recall[critical_idx]:.3f} (Most important!)")

print("\n📱 Mobile Deployment:")
print(f"  Model Size:      {tflite_size:.2f} MB")
print(f"  Inference Time:  {latency:.2f} ms")
print(f"  Ready for Flutter: ✓")

print("\n🚀 Next Steps:")
print("  1. Copy TFLite model to: frontend/mobile/android/assets/")
print("  2. Integrate with Flutter using tflite_flutter package")
print("  3. Test on real device (Android/iOS)")
print("  4. Deploy backend API with full Keras model")
print("  5. Monitor model performance with real patient data")

print("\n" + "="*70)